# Minimal end-to-end RAG inference + evaluation demo

This notebook is a **from-scratch, notebook-friendly RAG demo** without LangChain/LlamaIndex.

It includes:

- raw markdown document loading
- benchmark / qrels loading
- section-aware fixed chunking
- dense retrieval with a Hugging Face embedding model
- optional reranking with a Hugging Face reranker
- answer generation with a Hugging Face causal LM
- retrieval evaluation and generation evaluation

The notebook is written so it can **run end-to-end on a tiny demo dataset** first, then be swapped to your own markdown corpus and benchmark files.

## Libraries used
- `transformers` for generation and optional reranking
- `sentence-transformers` for dense embedding retrieval
- `torch`, `numpy`, `pandas`

> Yes — Hugging Face libraries can be used directly for both **embedding** and **generation** in this pipeline.


## 0. Optional installation

Run this cell only if the environment does not already have the required packages.


In [1]:
# Uncomment if needed
# %pip install -U transformers sentence-transformers accelerate scikit-learn pandas numpy torch


## 1. Setup

In [2]:

from __future__ import annotations

import os
import re
import json
import math
import random
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [3]:

# -------- paths --------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "demo_data"
RAW_DOC_DIR = DATA_DIR / "raw_docs"
BENCHMARK_DIR = DATA_DIR / "benchmark"

OUTPUT_DIR = PROJECT_ROOT / "demo_outputs"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
RETRIEVAL_DIR = OUTPUT_DIR / "retrieval"
GENERATION_DIR = OUTPUT_DIR / "generation"
EVAL_DIR = OUTPUT_DIR / "evaluation"

for p in [DATA_DIR, RAW_DOC_DIR, BENCHMARK_DIR, OUTPUT_DIR, ARTIFACT_DIR, RETRIEVAL_DIR, GENERATION_DIR, EVAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_DOC_INDEX_PATH = DATA_DIR / "raw_docs_index.jsonl"
BENCHMARK_QA_PATH = BENCHMARK_DIR / "benchmark_qa.jsonl"
QRELS_PATH = BENCHMARK_DIR / "qrels.jsonl"


In [4]:

# -------- base config --------
CONFIG = {
    "debug": True,
    "active_split": "dev",
    "eval_k_values": [1, 3, 5],
    "main_k": 5,

    # chunking
    "chunk_method": "fixed_sentence_window",
    "target_chunk_chars": 280,
    "chunk_overlap_sentences": 1,
    "min_chunk_chars": 80,
    "max_chunk_chars": 420,

    # retrieval
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "retrieval_top_k": 5,

    # optional reranking
    "use_reranker": False,
    "reranker_model_name": "BAAI/bge-reranker-base",
    "rerank_top_k": 5,

    # generation
    "generator_model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "max_new_tokens": 64,
    "temperature": 0.0,
    "do_sample": False,
    "context_top_n": 3,
}
CONFIG


{'debug': True,
 'active_split': 'dev',
 'eval_k_values': [1, 3, 5],
 'main_k': 5,
 'chunk_method': 'fixed_sentence_window',
 'target_chunk_chars': 280,
 'chunk_overlap_sentences': 1,
 'min_chunk_chars': 80,
 'max_chunk_chars': 420,
 'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'retrieval_top_k': 5,
 'use_reranker': False,
 'reranker_model_name': 'BAAI/bge-reranker-base',
 'rerank_top_k': 5,
 'generator_model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'max_new_tokens': 64,
 'temperature': 0.0,
 'do_sample': False,
 'context_top_n': 3}

## 2. Create a tiny markdown demo dataset

If you already have your own markdown corpus and benchmark files, you can skip this section and point the paths above to your own files.


In [5]:

def write_jsonl(path: Path, rows: List[Dict[str, Any]]):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Tiny demo markdown corpus
docs = {
    "doc_0001.md": """# Laksa

## Origins
Laksa is a spicy noodle dish commonly associated with Southeast Asia. It is especially linked to Peranakan culinary traditions.

## Ingredients
Many versions of laksa use coconut milk and noodles. Some regional versions are sourer and fish-based.

## Regional Variants
Popular variants include curry laksa and asam laksa.
""",
    "doc_0002.md": """# Rendang

## Background
Rendang is a slow-cooked meat dish originating from the Minangkabau culture of West Sumatra in Indonesia.

## Cooking Method
It is traditionally cooked for a long time with coconut milk and spices until the sauce is reduced.

## Cultural Role
Rendang is often served during ceremonies and festive occasions.
""",
    "doc_0003.md": """# Sushi

## History
Sushi is strongly associated with Japan. Modern sushi developed from older forms of preserved fish and rice dishes.

## Ingredients
Common sushi ingredients include vinegared rice, seafood, and vegetables.

## Serving Styles
Styles include nigiri, maki, and temaki.
"""
}

index_rows = []
for i, (fname, text) in enumerate(docs.items(), start=1):
    doc_id = f"doc_{i:04d}"
    path = RAW_DOC_DIR / fname
    path.write_text(text, encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()
    index_rows.append({
        "doc_id": doc_id,
        "file_path": str(path),
        "title": title,
        "source": "demo_local",
        "category": "food_history",
        "language": "en",
        "metadata": {}
    })

write_jsonl(RAW_DOC_INDEX_PATH, index_rows)

benchmark_rows = [
    {
        "query_id": "q_000001",
        "question": "Which region is laksa commonly associated with?",
        "answer": "Southeast Asia",
        "answer_type": "factoid",
        "gold_doc_ids": ["doc_0001"],
        "gold_evidence": [
            {
                "evidence_id": "ev_000001",
                "doc_id": "doc_0001",
                "section_path": ["Origins"],
                "start_char": 0,
                "end_char": 120,
                "evidence_text": "Laksa is a spicy noodle dish commonly associated with Southeast Asia."
            }
        ],
        "metadata": {"verified": True, "split": "dev"}
    },
    {
        "query_id": "q_000002",
        "question": "Which culture is rendang said to originate from?",
        "answer": "The Minangkabau culture of West Sumatra in Indonesia",
        "answer_type": "factoid",
        "gold_doc_ids": ["doc_0002"],
        "gold_evidence": [
            {
                "evidence_id": "ev_000002",
                "doc_id": "doc_0002",
                "section_path": ["Background"],
                "start_char": 0,
                "end_char": 160,
                "evidence_text": "Rendang is a slow-cooked meat dish originating from the Minangkabau culture of West Sumatra in Indonesia."
            }
        ],
        "metadata": {"verified": True, "split": "dev"}
    },
    {
        "query_id": "q_000003",
        "question": "Which country is sushi strongly associated with?",
        "answer": "Japan",
        "answer_type": "factoid",
        "gold_doc_ids": ["doc_0003"],
        "gold_evidence": [
            {
                "evidence_id": "ev_000003",
                "doc_id": "doc_0003",
                "section_path": ["History"],
                "start_char": 0,
                "end_char": 120,
                "evidence_text": "Sushi is strongly associated with Japan."
            }
        ],
        "metadata": {"verified": True, "split": "dev"}
    }
]
write_jsonl(BENCHMARK_QA_PATH, benchmark_rows)

qrels_rows = [
    {"query_id": row["query_id"], "gold_doc_ids": row["gold_doc_ids"], "gold_evidence_ids": [ev["evidence_id"] for ev in row["gold_evidence"]]}
    for row in benchmark_rows
]
write_jsonl(QRELS_PATH, qrels_rows)

print("Demo files written.")
print(RAW_DOC_INDEX_PATH)
print(BENCHMARK_QA_PATH)
print(QRELS_PATH)


Demo files written.
C:\Code\Text-Into-Meaning-RAG\demo_data\raw_docs_index.jsonl
C:\Code\Text-Into-Meaning-RAG\demo_data\benchmark\benchmark_qa.jsonl
C:\Code\Text-Into-Meaning-RAG\demo_data\benchmark\qrels.jsonl


## 3. Load raw markdown docs and benchmark

In [6]:

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_text(text: str) -> str:
    text = text.replace("\ufeff", "")
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def markdown_to_plain_text(md_text: str) -> str:
    text = re.sub(r"```.*?```", " ", md_text, flags=re.DOTALL)
    text = re.sub(r"`([^`]*)`", r"\1", text)
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)
    text = re.sub(r"^\s{0,3}#{1,6}\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\d+\.\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"[>*_~]", "", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return normalize_text(text)

def extract_markdown_sections(md_text: str) -> List[Dict[str, Any]]:
    lines = md_text.splitlines(keepends=True)
    sections = []
    current_path = []
    buffer = []
    start_char = 0
    cursor = 0

    def flush_section(end_char: int):
        nonlocal buffer, start_char
        text = "".join(buffer).strip()
        if text:
            sections.append({
                "section_path": current_path.copy(),
                "start_char": start_char,
                "end_char": end_char,
                "markdown_text": text
            })
        buffer = []
        start_char = end_char

    for line in lines:
        heading = re.match(r"^\s*(#{1,6})\s+(.*)$", line.strip("\n"))
        if heading:
            if buffer:
                flush_section(cursor)
            level = len(heading.group(1))
            title = heading.group(2).strip()
            current_path = current_path[:level-1] + [title]
            start_char = cursor
        buffer.append(line)
        cursor += len(line)

    if buffer:
        flush_section(cursor)
    return sections

def simple_sentence_split(text: str) -> List[str]:
    # lightweight splitter suitable for demo purposes
    text = text.strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    return [p.strip() for p in parts if p.strip()]

raw_doc_index = read_jsonl(RAW_DOC_INDEX_PATH)
raw_docs = []
raw_doc_by_id = {}

for row in raw_doc_index:
    file_path = Path(row["file_path"])
    md_text = normalize_text(file_path.read_text(encoding="utf-8"))
    plain_text = markdown_to_plain_text(md_text)
    sections = extract_markdown_sections(md_text)

    doc = {
        **row,
        "markdown_text": md_text,
        "plain_text": plain_text,
        "sections": sections,
    }
    raw_docs.append(doc)
    raw_doc_by_id[row["doc_id"]] = doc

benchmark_samples = read_jsonl(BENCHMARK_QA_PATH)
qrels_rows = read_jsonl(QRELS_PATH)
qrels_by_query_id = {row["query_id"]: row for row in qrels_rows}

active_queries = [row for row in benchmark_samples if row.get("metadata", {}).get("split") == CONFIG["active_split"]]
print("raw docs:", len(raw_docs))
print("benchmark queries:", len(active_queries))
print("sample doc ids:", [d["doc_id"] for d in raw_docs])


raw docs: 3
benchmark queries: 3
sample doc ids: ['doc_0001', 'doc_0002', 'doc_0003']


In [7]:

# Sanity checks
display(pd.DataFrame([{
    "doc_id": d["doc_id"],
    "title": d["title"],
    "plain_chars": len(d["plain_text"]),
    "num_sections": len(d["sections"])
} for d in raw_docs]))

active_queries[:2]


,doc_id,title,plain_chars,num_sections
0,doc_0001,Laksa,327,4
1,doc_0002,Rendang,318,4
2,doc_0003,Sushi,271,4


[{'query_id': 'q_000001',
  'question': 'Which region is laksa commonly associated with?',
  'answer': 'Southeast Asia',
  'answer_type': 'factoid',
  'gold_doc_ids': ['doc_0001'],
  'gold_evidence': [{'evidence_id': 'ev_000001',
    'doc_id': 'doc_0001',
    'section_path': ['Origins'],
    'start_char': 0,
    'end_char': 120,
    'evidence_text': 'Laksa is a spicy noodle dish commonly associated with Southeast Asia.'}],
  'metadata': {'verified': True, 'split': 'dev'}},
 {'query_id': 'q_000002',
  'question': 'Which culture is rendang said to originate from?',
  'answer': 'The Minangkabau culture of West Sumatra in Indonesia',
  'answer_type': 'factoid',
  'gold_doc_ids': ['doc_0002'],
  'gold_evidence': [{'evidence_id': 'ev_000002',
    'doc_id': 'doc_0002',
    'section_path': ['Background'],
    'start_char': 0,
    'end_char': 160,
    'evidence_text': 'Rendang is a slow-cooked meat dish originating from the Minangkabau culture of West Sumatra in Indonesia.'}],
  'metadata': {'v

## 4. Build base units and fixed chunks

In [8]:

@dataclass
class BaseUnit:
    unit_id: str
    doc_id: str
    text: str
    start_char: int
    end_char: int
    section_path: List[str]

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    file_path: str
    chunk_method: str
    text: str
    start_char: int
    end_char: int
    section_path: List[str]
    metadata: Dict[str, Any]

def find_span(full_text: str, sub_text: str, start_pos: int = 0) -> Tuple[int, int]:
    idx = full_text.find(sub_text, start_pos)
    if idx == -1:
        idx = start_pos
        end_idx = min(len(full_text), start_pos + len(sub_text))
        return idx, end_idx
    return idx, idx + len(sub_text)

def build_base_units(raw_docs: List[Dict[str, Any]]) -> List[BaseUnit]:
    units = []
    for doc in raw_docs:
        unit_counter = 0
        for sec in doc["sections"]:
            sec_plain = markdown_to_plain_text(sec["markdown_text"])
            sentences = simple_sentence_split(sec_plain)
            cursor = 0
            for sent in sentences:
                start_char, end_char = find_span(doc["plain_text"], sent, cursor)
                cursor = end_char
                units.append(BaseUnit(
                    unit_id=f'{doc["doc_id"]}_u_{unit_counter:04d}',
                    doc_id=doc["doc_id"],
                    text=sent,
                    start_char=start_char,
                    end_char=end_char,
                    section_path=sec["section_path"] if sec["section_path"] else [doc["title"]]
                ))
                unit_counter += 1
    return units

base_units = build_base_units(raw_docs)
base_units_by_doc = defaultdict(list)
for u in base_units:
    base_units_by_doc[u.doc_id].append(u)

print("base units:", len(base_units))
display(pd.DataFrame([asdict(u) for u in base_units[:8]]))


base units: 15


,unit_id,doc_id,text,start_char,end_char,section_path
0,doc_0001_u_0000,doc_0001,Laksa,0,5,[Laksa]
1,doc_0001_u_0001,doc_0001,Origins\nLaksa is a spicy noodle dish commonly...,6,83,"[Laksa, Origins]"
2,doc_0001_u_0002,doc_0001,It is especially linked to Peranakan culinary ...,84,141,"[Laksa, Origins]"
3,doc_0001_u_0003,doc_0001,Ingredients\nMany versions of laksa use coconu...,142,206,"[Laksa, Ingredients]"
4,doc_0001_u_0004,doc_0001,Some regional versions are sourer and fish-based.,207,256,"[Laksa, Ingredients]"
5,doc_0001_u_0005,doc_0001,Regional Variants\nPopular variants include cu...,257,327,"[Laksa, Regional Variants]"
6,doc_0002_u_0000,doc_0002,Rendang,0,7,[Rendang]
7,doc_0002_u_0001,doc_0002,Background\nRendang is a slow-cooked meat dish...,8,124,"[Rendang, Background]"


In [9]:

def build_fixed_chunks(
    base_units_by_doc: Dict[str, List[BaseUnit]],
    raw_doc_by_id: Dict[str, Dict[str, Any]],
    target_chunk_chars: int = 280,
    overlap_sentences: int = 1,
    min_chunk_chars: int = 80,
    max_chunk_chars: int = 420,
) -> List[Chunk]:
    chunks = []
    for doc_id, units in base_units_by_doc.items():
        i = 0
        chunk_idx = 0
        while i < len(units):
            current_units = []
            total_chars = 0
            j = i
            while j < len(units):
                cand_len = len(units[j].text)
                if current_units and total_chars + cand_len > target_chunk_chars:
                    break
                current_units.append(units[j])
                total_chars += cand_len + 1
                if total_chars >= target_chunk_chars:
                    break
                j += 1

            if not current_units:
                current_units = [units[i]]

            text = " ".join(u.text for u in current_units).strip()
            if len(text) < min_chunk_chars and j + 1 < len(units):
                # greedily extend
                k = j + 1
                while k < len(units) and len(text) < min_chunk_chars and len(text) < max_chunk_chars:
                    current_units.append(units[k])
                    text = " ".join(u.text for u in current_units).strip()
                    k += 1

            start_char = current_units[0].start_char
            end_char = current_units[-1].end_char
            section_path = current_units[0].section_path

            chunks.append(Chunk(
                chunk_id=f"{doc_id}_chunk_{chunk_idx:04d}",
                doc_id=doc_id,
                file_path=raw_doc_by_id[doc_id]["file_path"],
                chunk_method="fixed_sentence_window",
                text=text,
                start_char=start_char,
                end_char=end_char,
                section_path=section_path,
                metadata={
                    "chunk_index": chunk_idx,
                    "char_count": len(text),
                    "num_units": len(current_units),
                    "target_chunk_chars": target_chunk_chars,
                }
            ))
            chunk_idx += 1

            next_i = max(i + 1, len(current_units) + i - overlap_sentences)
            i = next_i
    return chunks

chunks = build_fixed_chunks(
    base_units_by_doc,
    raw_doc_by_id,
    target_chunk_chars=CONFIG["target_chunk_chars"],
    overlap_sentences=CONFIG["chunk_overlap_sentences"],
    min_chunk_chars=CONFIG["min_chunk_chars"],
    max_chunk_chars=CONFIG["max_chunk_chars"],
)
chunk_by_id = {c.chunk_id: c for c in chunks}
doc_to_chunks = defaultdict(list)
for c in chunks:
    doc_to_chunks[c.doc_id].append(c)

chunk_df = pd.DataFrame([{
    "chunk_id": c.chunk_id,
    "doc_id": c.doc_id,
    "chars": len(c.text),
    "start_char": c.start_char,
    "end_char": c.end_char,
    "text_preview": c.text[:90]
} for c in chunks])
display(chunk_df)
print("num chunks:", len(chunks))
print("avg chars:", round(chunk_df["chars"].mean(), 2))


,chunk_id,doc_id,chars,start_char,end_char,text_preview
0,doc_0001_chunk_0000,doc_0001,256,0,256,Laksa Origins\nLaksa is a spicy noodle dish co...
1,doc_0001_chunk_0001,doc_0001,120,207,327,Some regional versions are sourer and fish-bas...
2,doc_0001_chunk_0002,doc_0001,70,257,327,Regional Variants\nPopular variants include cu...
3,doc_0002_chunk_0000,doc_0002,239,0,239,Rendang Background\nRendang is a slow-cooked m...
4,doc_0002_chunk_0001,doc_0002,193,125,318,Cooking Method\nIt is traditionally cooked for...
5,doc_0002_chunk_0002,doc_0002,78,240,318,Cultural Role\nRendang is often served during ...
6,doc_0003_chunk_0000,doc_0003,271,0,271,Sushi History\nSushi is strongly associated wi...
7,doc_0003_chunk_0001,doc_0003,55,216,271,"Serving Styles\nStyles include nigiri, maki, a..."


num chunks: 8
avg chars: 160.25


In [10]:

# save chunk artifact
run_name = f'{CONFIG["chunk_method"]}__dense__minilm__{"rerank" if CONFIG["use_reranker"] else "none"}'
chunk_artifact_path = ARTIFACT_DIR / f"{run_name}_chunks.jsonl"

with open(chunk_artifact_path, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")

config_artifact_path = ARTIFACT_DIR / f"{run_name}_config.json"
with open(config_artifact_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

print("saved:", chunk_artifact_path)
print("saved:", config_artifact_path)


saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\artifacts\fixed_sentence_window__dense__minilm__none_chunks.jsonl
saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\artifacts\fixed_sentence_window__dense__minilm__none_config.json


## 5. Dense retrieval with Hugging Face embeddings

This section uses a Hugging Face embedding model through `sentence-transformers`.


In [11]:
from sentence_transformers import SentenceTransformer


C:\Users\25247\anaconda3\envs\trim_rag\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\25247\anaconda3\envs\trim_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:

embedding_model = SentenceTransformer(CONFIG["embedding_model_name"], device=device)

chunk_texts = [c.text for c in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_texts = [q["question"] for q in active_queries]
query_embeddings = embedding_model.encode(
    query_texts,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("chunk embeddings shape:", chunk_embeddings.shape)
print("query embeddings shape:", query_embeddings.shape)


chunk embeddings shape: (8, 384)
query embeddings shape: (3, 384)


In [13]:
def cosine_top_k(query_vec: np.ndarray, doc_matrix: np.ndarray, top_k: int) -> Tuple[np.ndarray, np.ndarray]:
    scores = doc_matrix @ query_vec
    idx = np.argsort(-scores)[:top_k]
    return idx, scores[idx]

first_stage_results = []

for q, qvec in zip(active_queries, query_embeddings):
    idxs, scores = cosine_top_k(qvec, chunk_embeddings, CONFIG["retrieval_top_k"])
    candidates = []
    for rank, (idx, score) in enumerate(zip(idxs, scores), start=1):
        c = chunks[int(idx)]
        candidates.append({
            "rank": rank,
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "score": float(score),
            "text": c.text,
            "start_char": c.start_char,
            "end_char": c.end_char,
        })
    first_stage_results.append({
        "query_id": q["query_id"],
        "question": q["question"],
        "run_name": run_name,
        "retriever": {
            "type": "dense",
            "embedding_model": CONFIG["embedding_model_name"],
            "top_k": CONFIG["retrieval_top_k"],
        },
        "candidates": candidates,
    })

first_stage_path = RETRIEVAL_DIR / f"{run_name}_first_stage.jsonl"
with open(first_stage_path, "w", encoding="utf-8") as f:
    for row in first_stage_results:
        f.write(json.dumps(row, ensure_ascii=False) + "\\n")

print("saved:", first_stage_path)
first_stage_results[0]


saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\retrieval\fixed_sentence_window__dense__minilm__none_first_stage.jsonl


{'query_id': 'q_000001',
 'question': 'Which region is laksa commonly associated with?',
 'run_name': 'fixed_sentence_window__dense__minilm__none',
 'retriever': {'type': 'dense',
  'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
  'top_k': 5},
 'candidates': [{'rank': 1,
   'chunk_id': 'doc_0001_chunk_0000',
   'doc_id': 'doc_0001',
   'score': 0.6613536477088928,
   'text': 'Laksa Origins\nLaksa is a spicy noodle dish commonly associated with Southeast Asia. It is especially linked to Peranakan culinary traditions. Ingredients\nMany versions of laksa use coconut milk and noodles. Some regional versions are sourer and fish-based.',
   'start_char': 0,
   'end_char': 256},
  {'rank': 2,
   'chunk_id': 'doc_0001_chunk_0002',
   'doc_id': 'doc_0001',
   'score': 0.6123132705688477,
   'text': 'Regional Variants\nPopular variants include curry laksa and asam laksa.',
   'start_char': 257,
   'end_char': 327},
  {'rank': 3,
   'chunk_id': 'doc_0001_chunk_0001',
   'doc_id': 'd

## 6. Optional Hugging Face reranking

This section is optional. Set `CONFIG["use_reranker"] = True` to enable it.


In [14]:

reranked_results = None

if CONFIG["use_reranker"]:
    from sentence_transformers import CrossEncoder

    reranker = CrossEncoder(CONFIG["reranker_model_name"], device=device)
    reranked_results = []

    for row in first_stage_results:
        pairs = [(row["question"], cand["text"]) for cand in row["candidates"]]
        rerank_scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)

        enriched = []
        for cand, rscore in zip(row["candidates"], rerank_scores):
            item = dict(cand)
            item["original_rank"] = item["rank"]
            item["retrieval_score"] = item["score"]
            item["rerank_score"] = float(rscore)
            enriched.append(item)

        enriched = sorted(enriched, key=lambda x: x["rerank_score"], reverse=True)
        for new_rank, item in enumerate(enriched[:CONFIG["rerank_top_k"]], start=1):
            item["final_rank"] = new_rank

        reranked_results.append({
            "query_id": row["query_id"],
            "question": row["question"],
            "run_name": run_name,
            "first_stage_retriever": row["retriever"],
            "reranker": {
                "type": "cross_encoder",
                "model_name": CONFIG["reranker_model_name"],
                "input_top_k": len(row["candidates"]),
                "output_top_n": CONFIG["rerank_top_k"],
            },
            "candidates": enriched[:CONFIG["rerank_top_k"]],
        })

    rerank_path = RETRIEVAL_DIR / f"{run_name}_reranked.jsonl"
    with open(rerank_path, "w", encoding="utf-8") as f:
        for row in reranked_results:
            f.write(json.dumps(row, ensure_ascii=False) + "\\n")
    print("saved:", rerank_path)
else:
    print("Reranker disabled. Using first-stage retrieval results directly.")


Reranker disabled. Using first-stage retrieval results directly.


## 7. Retrieval evaluation

This demo evaluates retrieval mainly at the **doc level**:
a query is counted as correct if the returned chunk comes from a gold document.


In [15]:

def dcg(relevances: List[int]) -> float:
    total = 0.0
    for i, rel in enumerate(relevances, start=1):
        total += rel / math.log2(i + 1)
    return total

def ndcg_at_k(relevances: List[int], k: int) -> float:
    rels = relevances[:k]
    actual = dcg(rels)
    ideal = dcg(sorted(rels, reverse=True))
    return actual / ideal if ideal > 0 else 0.0

def evaluate_retrieval(results: List[Dict[str, Any]], benchmark_by_qid: Dict[str, Dict[str, Any]], k_values: List[int]):
    per_query = []
    summary_acc = {f"recall@{k}": [] for k in k_values}
    summary_acc.update({f"mrr@{k}": [] for k in k_values})
    summary_acc.update({f"ndcg@{k}": [] for k in k_values})

    for row in results:
        qid = row["query_id"]
        gold_docs = set(benchmark_by_qid[qid]["gold_doc_ids"])
        retrieved_docs = [cand["doc_id"] for cand in row["candidates"]]
        relevances = [1 if d in gold_docs else 0 for d in retrieved_docs]

        first_relevant_rank = None
        for idx, rel in enumerate(relevances, start=1):
            if rel == 1:
                first_relevant_rank = idx
                break

        per_row = {"query_id": qid, "first_relevant_rank": first_relevant_rank}
        for k in k_values:
            hit = int(any(rel == 1 for rel in relevances[:k]))
            rr = 0.0
            if first_relevant_rank is not None and first_relevant_rank <= k:
                rr = 1.0 / first_relevant_rank
            nd = ndcg_at_k(relevances, k)
            per_row[f"hit@{k}"] = hit
            per_row[f"recall@{k}"] = hit
            per_row[f"mrr@{k}"] = rr
            per_row[f"ndcg@{k}"] = nd

            summary_acc[f"recall@{k}"].append(hit)
            summary_acc[f"mrr@{k}"].append(rr)
            summary_acc[f"ndcg@{k}"].append(nd)
        per_query.append(per_row)

    summary = {metric: float(np.mean(vals)) if vals else 0.0 for metric, vals in summary_acc.items()}
    return {"summary": summary, "per_query": per_query}

benchmark_by_qid = {row["query_id"]: row for row in active_queries}
retrieval_input = reranked_results if reranked_results is not None else first_stage_results
retrieval_eval = evaluate_retrieval(retrieval_input, benchmark_by_qid, CONFIG["eval_k_values"])
retrieval_eval


{'summary': {'recall@1': 1.0,
  'recall@3': 1.0,
  'recall@5': 1.0,
  'mrr@1': 1.0,
  'mrr@3': 1.0,
  'mrr@5': 1.0,
  'ndcg@1': 1.0,
  'ndcg@3': 1.0,
  'ndcg@5': 0.9501149685115848},
 'per_query': [{'query_id': 'q_000001',
   'first_relevant_rank': 1,
   'hit@1': 1,
   'recall@1': 1,
   'mrr@1': 1.0,
   'ndcg@1': 1.0,
   'hit@3': 1,
   'recall@3': 1,
   'mrr@3': 1.0,
   'ndcg@3': 1.0,
   'hit@5': 1,
   'recall@5': 1,
   'mrr@5': 1.0,
   'ndcg@5': 1.0},
  {'query_id': 'q_000002',
   'first_relevant_rank': 1,
   'hit@1': 1,
   'recall@1': 1,
   'mrr@1': 1.0,
   'ndcg@1': 1.0,
   'hit@3': 1,
   'recall@3': 1,
   'mrr@3': 1.0,
   'ndcg@3': 1.0,
   'hit@5': 1,
   'recall@5': 1,
   'mrr@5': 1.0,
   'ndcg@5': 1.0},
  {'query_id': 'q_000003',
   'first_relevant_rank': 1,
   'hit@1': 1,
   'recall@1': 1,
   'mrr@1': 1.0,
   'ndcg@1': 1.0,
   'hit@3': 1,
   'recall@3': 1,
   'mrr@3': 1.0,
   'ndcg@3': 1.0,
   'hit@5': 1,
   'recall@5': 1,
   'mrr@5': 1.0,
   'ndcg@5': 0.8503449055347546}]}

In [16]:

retrieval_metrics_path = EVAL_DIR / f"{run_name}_retrieval_metrics.json"
with open(retrieval_metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "run_name": run_name,
        "evaluation_level": "doc",
        "dataset_split": CONFIG["active_split"],
        "summary": retrieval_eval["summary"],
        "per_query": retrieval_eval["per_query"],
    }, f, ensure_ascii=False, indent=2)

print("saved:", retrieval_metrics_path)
display(pd.DataFrame(retrieval_eval["per_query"]))


saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\evaluation\fixed_sentence_window__dense__minilm__none_retrieval_metrics.json


,query_id,first_relevant_rank,hit@1,recall@1,mrr@1,ndcg@1,hit@3,recall@3,mrr@3,ndcg@3,hit@5,recall@5,mrr@5,ndcg@5
0,q_000001,1,1,1,1.0,1.0,1,1,1.0,1.0,1,1,1.0,1.000000
1,q_000002,1,1,1,1.0,1.0,1,1,1.0,1.0,1,1,1.0,1.000000
2,q_000003,1,1,1,1.0,1.0,1,1,1.0,1.0,1,1,1.0,0.850345


## 8. Generation with a Hugging Face causal LM

This section uses `transformers` directly for answer generation.


In [17]:

from transformers import AutoTokenizer, AutoModelForCausalLM


In [18]:

tokenizer = AutoTokenizer.from_pretrained(CONFIG["generator_model_name"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gen_dtype = torch.float16 if device == "cuda" else torch.float32
generator_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["generator_model_name"],
    torch_dtype=gen_dtype,
    trust_remote_code=True,
)
generator_model.to(device)
generator_model.eval()

print("generator loaded:", CONFIG["generator_model_name"])


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generator loaded: Qwen/Qwen2.5-0.5B-Instruct


In [19]:

def build_prompt(question: str, contexts: List[Dict[str, Any]]) -> str:
    context_block = "\n\n".join(
        [f"[Context {i}] {c['text']}" for i, c in enumerate(contexts, start=1)]
    )
    prompt = f"""You are a helpful QA assistant.
Answer the question using only the retrieved context.
If the answer is not supported by the context, say "I don't know based on the provided context."

Context:
{context_block}

Question: {question}

Answer:"""
    return prompt

def generate_answer(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=CONFIG["do_sample"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text


In [20]:

generation_results = []

for row in retrieval_input:
    qid = row["query_id"]
    q = benchmark_by_qid[qid]
    contexts = row["candidates"][:CONFIG["context_top_n"]]
    prompt = build_prompt(q["question"], contexts)
    answer = generate_answer(prompt)

    generation_results.append({
        "query_id": qid,
        "question": q["question"],
        "gold_answer": q["answer"],
        "run_name": run_name,
        "context_source": "reranked" if reranked_results is not None else "first_stage",
        "used_contexts": contexts,
        "prompt_metadata": {
            "num_contexts": len(contexts),
            "prompt_template": "qa_v1"
        },
        "generator": {
            "model_name": CONFIG["generator_model_name"],
            "max_new_tokens": CONFIG["max_new_tokens"],
            "temperature": CONFIG["temperature"],
        },
        "generated_answer": answer,
        "raw_output": answer,
    })

answers_path = GENERATION_DIR / f"{run_name}_answers.jsonl"
with open(answers_path, "w", encoding="utf-8") as f:
    for row in generation_results:
        f.write(json.dumps(row, ensure_ascii=False) + "\\n")

print("saved:", answers_path)
generation_results[0]


C:\Users\25247\anaconda3\envs\trim_rag\lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\25247\anaconda3\envs\trim_rag\lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\25247\anaconda3\envs\trim_rag\lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\generation\fixed_sentence_window__dense__minilm__none_answers.jsonl


{'query_id': 'q_000001',
 'question': 'Which region is laksa commonly associated with?',
 'gold_answer': 'Southeast Asia',
 'run_name': 'fixed_sentence_window__dense__minilm__none',
 'context_source': 'first_stage',
 'used_contexts': [{'rank': 1,
   'chunk_id': 'doc_0001_chunk_0000',
   'doc_id': 'doc_0001',
   'score': 0.6613536477088928,
   'text': 'Laksa Origins\nLaksa is a spicy noodle dish commonly associated with Southeast Asia. It is especially linked to Peranakan culinary traditions. Ingredients\nMany versions of laksa use coconut milk and noodles. Some regional versions are sourer and fish-based.',
   'start_char': 0,
   'end_char': 256},
  {'rank': 2,
   'chunk_id': 'doc_0001_chunk_0002',
   'doc_id': 'doc_0001',
   'score': 0.6123132705688477,
   'text': 'Regional Variants\nPopular variants include curry laksa and asam laksa.',
   'start_char': 257,
   'end_char': 327},
  {'rank': 3,
   'chunk_id': 'doc_0001_chunk_0001',
   'doc_id': 'doc_0001',
   'score': 0.590183854103088

## 9. Generation evaluation

In [21]:

def normalize_answer(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def exact_match(pred: str, gold: str) -> float:
    return float(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def lcs_len(x: List[str], y: List[str]) -> int:
    dp = [[0]*(len(y)+1) for _ in range(len(x)+1)]
    for i in range(1, len(x)+1):
        for j in range(1, len(y)+1):
            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def rouge_l_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    lcs = lcs_len(pred_tokens, gold_tokens)
    prec = lcs / len(pred_tokens)
    rec = lcs / len(gold_tokens)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

per_query_gen = []
for row in generation_results:
    em = exact_match(row["generated_answer"], row["gold_answer"])
    f1 = token_f1(row["generated_answer"], row["gold_answer"])
    rl = rouge_l_f1(row["generated_answer"], row["gold_answer"])
    per_query_gen.append({
        "query_id": row["query_id"],
        "exact_match": em,
        "token_f1": f1,
        "rouge_l_f1": rl,
        "gold_answer": row["gold_answer"],
        "generated_answer": row["generated_answer"],
    })

generation_eval = {
    "summary": {
        "exact_match": float(np.mean([r["exact_match"] for r in per_query_gen])),
        "token_f1": float(np.mean([r["token_f1"] for r in per_query_gen])),
        "rouge_l_f1": float(np.mean([r["rouge_l_f1"] for r in per_query_gen])),
    },
    "per_query": per_query_gen,
}
generation_eval


{'summary': {'exact_match': 0.0,
  'token_f1': 0.12729559748427674,
  'rouge_l_f1': 0.12729559748427674},
 'per_query': [{'query_id': 'q_000001',
   'exact_match': 0.0,
   'token_f1': 0.07999999999999999,
   'rouge_l_f1': 0.07999999999999999,
   'gold_answer': 'Southeast Asia',
   'generated_answer': 'Laksa is commonly associated with Peranakan culinary traditions in Southeast Asia. The context states that it is particularly linked to these regions. Therefore, based on the provided context, we can conclude that Laksa is associated with Peranakan culinary traditions in Southeast Asia.Human resources (HR) professionals have been working for'},
  {'query_id': 'q_000002',
   'exact_match': 0.0,
   'token_f1': 0.2641509433962264,
   'rouge_l_f1': 0.2641509433962264,
   'gold_answer': 'The Minangkabau culture of West Sumatra in Indonesia',
   'generated_answer': 'The culture that is said to originate from rendang is the Minangkabau culture of West Sumatra in Indonesia. Based on the informati

In [22]:

generation_metrics_path = EVAL_DIR / f"{run_name}_generation_metrics.json"
with open(generation_metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "run_name": run_name,
        "dataset_split": CONFIG["active_split"],
        "summary": generation_eval["summary"],
        "per_query": generation_eval["per_query"],
    }, f, ensure_ascii=False, indent=2)

print("saved:", generation_metrics_path)
display(pd.DataFrame(generation_eval["per_query"]))


saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\evaluation\fixed_sentence_window__dense__minilm__none_generation_metrics.json


,query_id,exact_match,token_f1,rouge_l_f1,gold_answer,generated_answer
0,q_000001,0.0,0.080000,0.080000,Southeast Asia,Laksa is commonly associated with Peranakan cu...
1,q_000002,0.0,0.264151,0.264151,The Minangkabau culture of West Sumatra in Ind...,The culture that is said to originate from ren...
2,q_000003,0.0,0.037736,0.037736,Japan,Japan\n\nThe context provides information abou...


## 10. Summary table

In [23]:

summary_row = {
    "run_name": run_name,
    "chunk_method": CONFIG["chunk_method"],
    "retriever": "dense",
    "embedding_model": CONFIG["embedding_model_name"],
    "reranker": CONFIG["reranker_model_name"] if CONFIG["use_reranker"] else "none",
    "generator": CONFIG["generator_model_name"],
    "split": CONFIG["active_split"],
}

for k, v in retrieval_eval["summary"].items():
    summary_row[k] = v
for k, v in generation_eval["summary"].items():
    summary_row[k] = v

summary_df = pd.DataFrame([summary_row])
summary_csv_path = EVAL_DIR / "all_runs_summary.csv"

if summary_csv_path.exists():
    old_df = pd.read_csv(summary_csv_path)
    summary_df = pd.concat([old_df, summary_df], ignore_index=True)

summary_df = summary_df.drop_duplicates(subset=["run_name", "split"], keep="last")
summary_df.to_csv(summary_csv_path, index=False)

print("saved:", summary_csv_path)
summary_df


saved: C:\Code\Text-Into-Meaning-RAG\demo_outputs\evaluation\all_runs_summary.csv


,run_name,chunk_method,retriever,embedding_model,reranker,generator,split,recall@1,recall@3,recall@5,mrr@1,mrr@3,mrr@5,ndcg@1,ndcg@3,ndcg@5,exact_match,token_f1,rouge_l_f1
1,fixed_sentence_window__dense__minilm__none,fixed_sentence_window,dense,sentence-transformers/all-MiniLM-L6-v2,none,Qwen/Qwen2.5-0.5B-Instruct,dev,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.950115,0.0,0.127296,0.127296


## 11. What to replace next

To move from this demo to your real coursework pipeline:

1. Replace the tiny demo markdown files with your own `raw_docs/*.md`
2. Replace `raw_docs_index.jsonl`, `benchmark_qa.jsonl`, and `qrels.jsonl`
3. Swap the chunking module for the methods you want to compare
4. Swap the embedding model and generator model as needed
5. Turn on the reranker if you want a two-stage retrieval setup


In [24]:
import sys
print(sys.version)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

try:
    import torchvision
    print("torchvision:", torchvision.__version__)
except Exception as e:
    print("torchvision import failed:", repr(e))

try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception as e:
    print("transformers import failed:", repr(e))

try:
    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__)
except Exception as e:
    print("sentence-transformers import failed:", repr(e))

3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
torch: 2.7.1+cu118
cuda available: True
torchvision: 0.22.1+cu118
transformers: 4.49.0
sentence-transformers: 5.3.0
